# LeetCode 831 — Masking Personal Information bằng Python

Link đề: https://leetcode.com/problems/masking-personal-information/ · Độ khó: **Medium** · Chủ đề: String (xử lý chuỗi, mô phỏng theo quy tắc)

Skeleton LeetCode:

```python
class Solution:
    def maskPII(self, s: str) -> str:
        ...
```

Ghi chú: bài không có thuật toán cao siêu; cái khó là **đọc kỹ quy tắc** và xử lý chuỗi cho gọn, không sót trường hợp (tên email chỉ 2 chữ cái vẫn có 5 dấu `*`; số điện thoại có nhiều kiểu ký tự phân cách; mã quốc gia 0–3 chữ số). Notebook đi từ cách tự duyệt từng ký tự, qua các hàm chuỗi của Python, biểu thức chính quy (regex), và cách dùng bảng tra.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: chuỗi `s` là **email** hoặc **số điện thoại**; email thì che phần giữa của tên bằng 5 dấu `*` và đổi sang chữ thường; số điện thoại thì bỏ ký tự phân cách rồi chỉ giữ 4 chữ số cuối, kèm mã quốc gia che bằng `*`.
2. Nắm các công cụ Python cần dùng: `in`, `str.lower`, `str.split`, `str.isdigit`, cắt chuỗi với chỉ số âm, nhân chuỗi (`"*" * 3`), `str.join`, f-string, `str.translate`, và biểu thức chính quy `re` ở mức cơ bản.
3. Đi từ **duyệt từng ký tự bằng tay** → **dùng hàm chuỗi dựng sẵn** → **biểu thức chính quy** → **bảng tra theo độ dài mã quốc gia**, và biết vì sao các cách cho cùng kết quả.
4. Hiểu mấu chốt: **chia hai nhánh** bằng điều kiện đơn giản (`"@" in s`), rồi mỗi nhánh làm theo đúng quy tắc: email `name[0] + "*****" + name[-1] + "@" + domain.lower()`; số điện thoại: lấy các chữ số, `n = len(digits) - 10` là độ dài mã quốc gia, kết quả `PREFIX[n] + "***-***-" + digits[-4:]`.
5. Tránh các lỗi kinh điển: nhân số dấu `*` theo độ dài tên thay vì luôn 5, quên đổi chữ thường, lấy `s[-4:]` khi chưa bỏ ký tự phân cách, tính sai độ dài mã quốc gia, quên dấu `+`, và dùng `str.replace` che phần giữa sai khi tên có ký tự lặp.

## 2. Đề bài & ví dụ

Cho chuỗi thông tin cá nhân `s`, là một **địa chỉ email** hoặc một **số điện thoại**. Hãy trả về chuỗi đã **che (mask)** theo các quy tắc sau.

**Email.** Một email gồm:

- **tên (name)** gồm **ít nhất hai** chữ cái Anh (hoa hoặc thường), tiếp theo là
- ký tự `'@'`, tiếp theo là
- **miền (domain)** gồm chữ cái Anh (hoa hoặc thường) và một dấu chấm `'.'` ở **giữa** (không là ký tự đầu hoặc cuối).

Che email:

- Đổi mọi chữ hoa trong **tên** và **miền** thành chữ thường.
- Các chữ cái **ở giữa** của tên (tức mọi chữ trừ chữ đầu và chữ cuối) thay bằng đúng 5 dấu sao `"*****"`.

**Số điện thoại.** Số điện thoại có dạng:

- Gồm **10–13 chữ số**.
- **10 chữ số cuối** là **số nội hạt (local number)**.
- 0–3 chữ số còn lại ở đầu là **mã quốc gia**.
- Các chữ số được ngăn cách bằng một số **ký tự phân cách** thuộc tập `{'+', '-', '(', ')', ' '}`.

Che số điện thoại:

- Bỏ tất cả ký tự phân cách.
- Kết quả có dạng:
  - `"***-***-XXXX"` nếu mã quốc gia có 0 chữ số;
  - `"+*-***-***-XXXX"` nếu có 1 chữ số;
  - `"+**-***-***-XXXX"` nếu có 2 chữ số;
  - `"+***-***-***-XXXX"` nếu có 3 chữ số.
- `"XXXX"` là **4 chữ số cuối** của số nội hạt.

**Ví dụ 1**

- Input: `s = "LeetCode@LeetCode.com"`
- Output: `"l*****e@leetcode.com"`

**Ví dụ 2**

- Input: `s = "AB@qq.com"`
- Output: `"a*****b@qq.com"` (tên chỉ có 2 chữ cái nhưng vẫn có 5 dấu `*` ở giữa).

**Ví dụ 3**

- Input: `s = "1(234)567-890"`
- Output: `"***-***-7890"` (đúng 10 chữ số, mã quốc gia 0 chữ số).

**Ràng buộc:**

- `s` là email hợp lệ hoặc số điện thoại hợp lệ.
- Nếu là email: `8 <= s.length <= 40`; `s` gồm chữ cái Anh và đúng một `'@'`, đúng một `'.'`.
- Nếu là số điện thoại: `10 <= s.length <= 20`; `s` gồm chữ số, khoảng trắng và các ký hiệu `'('`, `')'`, `'-'`, `'+'`.

In [1]:
import re
import random

examples = [
    ("LeetCode@LeetCode.com", "l*****e@leetcode.com"),
    ("AB@qq.com", "a*****b@qq.com"),
    ("1(234)567-890", "***-***-7890"),
]
for s, expected in examples:
    print(repr(s), "->", repr(expected))

'LeetCode@LeetCode.com' -> 'l*****e@leetcode.com'
'AB@qq.com' -> 'a*****b@qq.com'
'1(234)567-890' -> '***-***-7890'


## 3. Kiến thức nền

### 3.1. Toán tử `in` với chuỗi: phân loại email / số điện thoại

`"@" in s` cho `True` nếu chuỗi `s` chứa ký tự (hoặc chuỗi con) `"@"`. Email luôn có đúng một `@`; số điện thoại chỉ gồm chữ số và các ký hiệu `+ - ( )` và khoảng trắng nên **không bao giờ có `@`**. Vậy một điều kiện đơn giản là đủ để rẽ nhánh.

(Một cách khác: ký tự đầu là chữ cái thì là email. Nhưng đừng dùng `s[0].isdigit()` để nhận biết số điện thoại — số điện thoại có thể bắt đầu bằng `+` hoặc `(`, xem lỗi ở mục 13.)

In [2]:
for s in ["AB@qq.com", "1(234)567-890", "+86(10)69 55 01 23 45"]:
    print(repr(s), "là email?", "@" in s)

'AB@qq.com' là email? True
'1(234)567-890' là email? False
'+86(10)69 55 01 23 45' là email? False


### 3.2. `lower()`, `split("@")` và cắt chuỗi với chỉ số âm

- `s.lower()` đổi chữ hoa thành chữ thường (không sửa `s` gốc, vì chuỗi bất biến; nó trả về chuỗi mới).
- `s.split("@")` tách chuỗi tại mỗi `@`, trả về danh sách: `"AB@qq.com".split("@")` là `["AB", "qq.com"]`. Ta có thể gán trực tiếp: `name, domain = s.split("@")`.
- Chỉ số **âm** đếm từ cuối: `name[-1]` là ký tự cuối, `name[0]` là ký tự đầu. Cắt `digits[-4:]` lấy 4 ký tự cuối, `digits[-10:]` lấy 10 ký tự cuối, `digits[:-10]` lấy phần còn lại phía trước.

In [3]:
s = "LeetCode@LeetCode.com"
name, domain = s.split("@")
print(name, "|", domain)
print(s.lower())
print(name[0], name[-1])               # L e

digits = "8610695501"
print(digits[-4:], digits[-10:], repr(digits[:-10]))   # 5501 8610695501 ''
digits = "12345678901234"
print(digits[:-10], "|", digits[-10:])                  # 1234 | 5678901234

LeetCode | LeetCode.com
leetcode@leetcode.com
L e
5501 8610695501 ''
1234 | 5678901234


### 3.3. Nhân chuỗi, `join`, f-string

- `"*" * 5` là `"*****"`; `"*" * 0` là chuỗi rỗng `""`. Rất tiện để tạo số dấu sao thay đổi.
- `"".join(list_chu)` nối các chuỗi trong danh sách thành một chuỗi.
- f-string: `f"{a}@{b}"` ghép giá trị biến vào chuỗi.

In [4]:
print("*" * 5, repr("*" * 0), "+" + "*" * 3 + "-")
print("".join(["a", "b", "c"]))
a, b = "x", "y.com"
print(f"{a}*****{a}@{b}")

***** '' +***-
abc
x*****x@y.com


### 3.4. Lọc chữ số: `isdigit()` và biểu thức sinh

`c.isdigit()` là `True` nếu ký tự `c` là chữ số `0..9`. Để giữ **chỉ các chữ số** trong chuỗi ta có:

```python
digits = "".join(c for c in s if c.isdigit())
```

(đây là một biểu thức sinh — generator expression — đưa vào `join`). Ví dụ `"1(234)567-890"` → `"1234567890"`.

In [5]:
s = "1(234)567-890"
digits = "".join(c for c in s if c.isdigit())
print(digits, len(digits))                 # 1234567890 10

s2 = "+86-(10)69 55 01 23"
d2 = "".join(c for c in s2 if c.isdigit())
print(d2, len(d2), "-> mã quốc gia có", len(d2) - 10, "chữ số")   # 861069550123 12 -> 2

1234567890 10
861069550123 12 -> mã quốc gia có 2 chữ số


### 3.5. `str.translate` để xoá ký tự phân cách

`str.maketrans("", "", "chars")` tạo bảng dịch: ký tự thuộc `"chars"` bị **xoá**. `s.translate(bảng)` áp dụng. Đây là cách xoá cả tập ký tự `+ - ( )` và khoảng trắng bằng một lệnh.

In [6]:
TABLE = str.maketrans("", "", "+-() ")
s = "+86 (10) 6955-0123"
print(s.translate(TABLE))

861069550123


### 3.6. Biểu thức chính quy `re` (rất ngắn gọn)

Module `re` cho phép mô tả "dạng chuỗi" bằng một mẫu (pattern):

- `\d` một chữ số, `\D` một ký tự **không** phải chữ số; `[A-Za-z]` một chữ cái; `+` "một hoặc nhiều lần"; `*` "không hoặc nhiều lần"; `{2,}` "ít nhất hai lần".
- `re.sub(mẫu, thay_bằng, s)` thay mọi chỗ khớp mẫu; `re.fullmatch(mẫu, s)` khớp **toàn bộ** chuỗi và trả về đối tượng (hoặc `None`); các cặp ngoặc `(...)` tạo **nhóm** lấy ra bằng `m.group(1)`.

Ví dụ: `re.sub(r"\D", "", "1(234)567-890")` xoá mọi ký tự không phải chữ số → `"1234567890"`.

In [7]:
print(re.sub(r"\D", "", "1(234)567-890"))                    # 1234567890
m = re.fullmatch(r"([A-Za-z])[A-Za-z]*([A-Za-z])@(.+)", "AB@qq.com")
print(m.groups())                                           # ('A', 'B', 'qq.com')
m = re.fullmatch(r"([A-Za-z])[A-Za-z]*([A-Za-z])@(.+)", "LeetCode@LeetCode.com")
print(m.groups())                                           # ('L', 'e', 'LeetCode.com')
print(re.fullmatch(r"\d+", "12a"))                          # None

1234567890
('A', 'B', 'qq.com')
('L', 'e', 'LeetCode.com')
None


## 4. Ý tưởng nền tảng / trực giác

Hãy xử lý từng ví dụ bằng tay.

**Email** `LeetCode@LeetCode.com`: tên `LeetCode` (8 chữ), miền `LeetCode.com`. Giữ chữ đầu `L` và chữ cuối `e`, thay giữa `eetCod` bằng `*****` → `L*****e`, rồi đổi chữ thường → `l*****e`; miền → `leetcode.com`. Kết quả `l*****e@leetcode.com`.

**Email** `AB@qq.com`: tên `AB` chỉ có 2 chữ; "giữa" là rỗng, nhưng quy tắc vẫn **chèn 5 dấu `*`** → `a*****b@qq.com`. Chú ý: số dấu `*` **không phụ thuộc độ dài tên**.

**Số điện thoại** `1(234)567-890`: bỏ ký tự phân cách → `1234567890` (10 chữ số). Mã quốc gia `10 - 10 = 0` chữ số → dạng `***-***-XXXX` với `XXXX` = 4 chữ số cuối `7890` → `***-***-7890`.

**Số điện thoại có mã quốc gia**, ví dụ `+86 (10) 6955-0123`: chữ số `861069550123` → 12 chữ số, mã quốc gia `12 - 10 = 2` chữ số (`86`) → `+**-***-***-0123`.

Kết luận: ta chỉ cần (1) phân loại, (2) email: giữ đầu/cuối + `*****`, (3) số điện thoại: lấy chữ số, đếm mã quốc gia, dựng chuỗi theo bảng.

In [8]:
digits = "861069550123"
country = len(digits) - 10
print("mã quốc gia có", country, "chữ số")
print("+" + "*" * country + "-" + "***-***-" + digits[-4:])      # +**-***-***-0123
digits0 = "1234567890"
print(repr("" if len(digits0) - 10 == 0 else "có mã"), "***-***-" + digits0[-4:])

mã quốc gia có 2 chữ số
+**-***-***-0123
'' ***-***-7890


## 5. Cách 1 — Brute Force: duyệt từng ký tự bằng tay

### Ý tưởng

Không dùng `lower`, `split`, `isdigit`... mà tự làm bằng vòng lặp:

1. Tìm vị trí `@` bằng vòng `for`. Nếu có: chia thành `name` và `domain` bằng cách tự cắt; đổi hoa → thường bằng `ord`/`chr` (chữ hoa `'A'..'Z'` có mã `65..90`, chữ thường hơn đúng `32`).
2. Nếu không có `@`: duyệt từng ký tự, **đếm và gom** những ký tự thuộc `'0'..'9'`.
3. Dựng kết quả bằng `if/elif` theo độ dài mã quốc gia.

Cách này **dài dòng** nhưng chỉ dùng kiến thức cơ bản nhất và là điểm xuất phát để hiểu mỗi hàm dựng sẵn thay thế cho đoạn nào.

### Code

In [9]:
def to_lower_char(c):
    # đổi một chữ hoa thành chữ thường bằng mã ASCII; ký tự khác giữ nguyên
    if "A" <= c <= "Z":
        return chr(ord(c) + 32)
    return c

def mask_pii_manual(s):
    # 1) tìm vị trí '@'
    at = -1
    for i in range(len(s)):
        if s[i] == "@":
            at = i
            break

    if at != -1:                                   # email
        name = ""
        for i in range(at):
            name += to_lower_char(s[i])
        domain = ""
        for i in range(at + 1, len(s)):
            domain += to_lower_char(s[i])
        return name[0] + "*****" + name[len(name) - 1] + "@" + domain

    # số điện thoại: gom chữ số
    digits = []
    for c in s:
        if "0" <= c <= "9":
            digits.append(c)
    total = len(digits)
    last4 = digits[total - 4] + digits[total - 3] + digits[total - 2] + digits[total - 1]
    country = total - 10
    if country == 0:
        return "***-***-" + last4
    elif country == 1:
        return "+*-***-***-" + last4
    elif country == 2:
        return "+**-***-***-" + last4
    else:
        return "+***-***-***-" + last4

for s, expected in examples:
    got = mask_pii_manual(s)
    print(got, expected, "OK" if got == expected else "SAI")

l*****e@leetcode.com l*****e@leetcode.com OK
a*****b@qq.com a*****b@qq.com OK
***-***-7890 ***-***-7890 OK


### Độ phức tạp và nút thắt

- **Time:** `O(n)` với `n = len(s)` — nhưng `name += ...` trong vòng lặp tạo chuỗi mới mỗi lần (cục bộ `O(n^2)` trong lý thuyết cho chuỗi dài; với `n ≤ 40` không ảnh hưởng).
- **Space:** `O(n)`.

**Nút thắt:** code dài, dễ nhầm chỉ số (`total - 4`...), nhiều `if/elif` lặp. Python đã có sẵn `lower`, `split`, `isdigit`, nhân chuỗi... giúp rút gọn nhiều. Hơn nữa 4 nhánh của mã quốc gia thực chất chỉ là **một công thức** theo `country`.

## 6. Ý tưởng thuật toán / bước đột phá

Hai nhận xét làm code gọn:

1. **Email chỉ cần ba mảnh:** `name[0]`, `name[-1]`, `domain`. Phần giữa của tên *không cần biết* — luôn thay bằng đúng `"*****"`. Hơn nữa, `lower()` có thể áp dụng cho **cả chuỗi** `s` ngay từ đầu (tên và miền đều đổi chữ thường, `@` và `.` không đổi).

2. **Số điện thoại: bốn trường hợp thực ra là một công thức.** Gọi `n = len(digits) - 10` (`0..3`) là độ dài mã quốc gia. Phần đầu của kết quả là:
   - `""` khi `n = 0`;
   - `"+" + "*" * n + "-"` khi `n ≥ 1`.

   Hoặc dùng **bảng tra**: `PREFIX = ["", "+*-", "+**-", "+***-"]`, kết quả `PREFIX[n] + "***-***-" + digits[-4:]`.

Bảng dưới liệt kê đầy đủ độ dài mã quốc gia và tiền tố.

In [10]:
PREFIX = ["", "+*-", "+**-", "+***-"]
for n in range(4):
    digits = "1234567890"[:0] + "9" * n + "5550001234"      # mã quốc gia gồm n chữ số 9
    print(f"n = {n}: số chữ số = {len(digits):>2} -> {PREFIX[n] + '***-***-' + digits[-4:]}")

n = 0: số chữ số = 10 -> ***-***-1234
n = 1: số chữ số = 11 -> +*-***-***-1234
n = 2: số chữ số = 12 -> +**-***-***-1234
n = 3: số chữ số = 13 -> +***-***-***-1234


## 7. Các cách tiếp theo

### 7.1. Cách 2 — Dùng hàm chuỗi dựng sẵn (lời giải nộp)

**Giải thích từng đoạn.**

1. `"@" in s` để rẽ nhánh.
2. Email: `s = s.lower()`, `name, domain = s.split("@")`, trả `name[0] + "*****" + name[-1] + "@" + domain`.
3. Số điện thoại: `digits = "".join(c for c in s if c.isdigit())`; `n = len(digits) - 10`; trả `PREFIX[n] + "***-***-" + digits[-4:]` (hoặc dựng tiền tố bằng `"+" + "*" * n + "-"` khi `n > 0`).

**Code demo:**

In [11]:
def mask_pii_builtin(s):
    if "@" in s:
        name, domain = s.lower().split("@")
        return name[0] + "*****" + name[-1] + "@" + domain
    digits = "".join(c for c in s if c.isdigit())
    n = len(digits) - 10
    prefix = "" if n == 0 else "+" + "*" * n + "-"
    return prefix + "***-***-" + digits[-4:]

for s, expected in examples:
    got = mask_pii_builtin(s)
    print(got, expected, "OK" if got == expected else "SAI")

l*****e@leetcode.com l*****e@leetcode.com OK
a*****b@qq.com a*****b@qq.com OK
***-***-7890 ***-***-7890 OK


**Code nộp LeetCode (Cách 2):**

In [12]:
class SolutionBuiltin:
    def maskPII(self, s: str) -> str:
        if "@" in s:
            name, domain = s.lower().split("@")
            return name[0] + "*****" + name[-1] + "@" + domain
        digits = "".join(c for c in s if c.isdigit())
        n = len(digits) - 10
        prefix = "" if n == 0 else "+" + "*" * n + "-"
        return prefix + "***-***-" + digits[-4:]

print(SolutionBuiltin().maskPII("AB@qq.com"))   # a*****b@qq.com

a*****b@qq.com


**Độ phức tạp (Cách 2):** Time `O(n)`, Space `O(n)`.

### 7.2. Cách 3 — Biểu thức chính quy

**Giải thích từng đoạn.**

1. **Email:** mẫu `([A-Za-z])[A-Za-z]*([A-Za-z])@(.+)` khớp **toàn chuỗi** và tách ra ba nhóm: chữ cái đầu của tên (nhóm 1), chữ cái cuối của tên (nhóm 2), miền (nhóm 3). `[A-Za-z]*` ở giữa nuốt phần giữa (có thể rỗng khi tên 2 chữ). Kết quả `f"{g1}*****{g2}@{g3}".lower()`.
2. **Số điện thoại:** `re.sub(r"\D", "", s)` xoá mọi ký tự **không** phải chữ số, ta được chuỗi chữ số; sau đó dựng kết quả như Cách 2.

Ưu điểm: mẫu *mô tả* dạng dữ liệu hợp lệ. Nhược điểm: khó đọc hơn với người mới; chậm hơn hàm chuỗi một chút (không đáng kể ở đây).

**Code demo:**

In [13]:
EMAIL_PATTERN = re.compile(r"([A-Za-z])[A-Za-z]*([A-Za-z])@(.+)")

def mask_pii_regex(s):
    m = EMAIL_PATTERN.fullmatch(s)
    if m:
        first, last, domain = m.groups()
        return f"{first}*****{last}@{domain}".lower()
    digits = re.sub(r"\D", "", s)
    n = len(digits) - 10
    prefix = "" if n == 0 else "+" + "*" * n + "-"
    return prefix + "***-***-" + digits[-4:]

for s, expected in examples:
    got = mask_pii_regex(s)
    print(got, expected, "OK" if got == expected else "SAI")

l*****e@leetcode.com l*****e@leetcode.com OK
a*****b@qq.com a*****b@qq.com OK
***-***-7890 ***-***-7890 OK


**Code nộp LeetCode (Cách 3):**

In [14]:
class SolutionRegex:
    def maskPII(self, s: str) -> str:
        m = re.fullmatch(r"([A-Za-z])[A-Za-z]*([A-Za-z])@(.+)", s)
        if m:
            first, last, domain = m.groups()
            return f"{first}*****{last}@{domain}".lower()
        digits = re.sub(r"\D", "", s)
        n = len(digits) - 10
        prefix = "" if n == 0 else "+" + "*" * n + "-"
        return prefix + "***-***-" + digits[-4:]

print(SolutionRegex().maskPII("1(234)567-890"))   # ***-***-7890

***-***-7890


**Độ phức tạp (Cách 3):** Time `O(n)` (mẫu đơn giản, không quay lui nặng), Space `O(n)`.

### 7.3. Cách 4 — Bảng tra + `translate`

**Giải thích từng đoạn.**

1. `str.translate` **xoá** ký tự phân cách `+ - ( )` và khoảng trắng chỉ trong một lệnh (thay cho `isdigit` + `join`).
2. Dùng bảng `PREFIX = ["", "+*-", "+**-", "+***-"]` tra theo `n = len(digits) - 10`: không còn `if`.
3. Email: `lower()` toàn chuỗi, `partition("@")` tách `(name, "@", domain)` (an toàn hơn `split` khi bạn muốn giữ lại `@`).

Cách này nhấn mạnh tư duy **hướng dữ liệu (table-driven)**: quy tắc nằm trong một bảng nhỏ thay vì chuỗi `if/elif`.

**Code demo:**

In [15]:
REMOVE_SEPARATORS = str.maketrans("", "", "+-() ")
PREFIX = ["", "+*-", "+**-", "+***-"]

def mask_pii_table(s):
    if "@" in s:
        name, at, domain = s.lower().partition("@")
        return name[0] + "*****" + name[-1] + at + domain
    digits = s.translate(REMOVE_SEPARATORS)
    return PREFIX[len(digits) - 10] + "***-***-" + digits[-4:]

for s, expected in examples:
    got = mask_pii_table(s)
    print(got, expected, "OK" if got == expected else "SAI")

l*****e@leetcode.com l*****e@leetcode.com OK
a*****b@qq.com a*****b@qq.com OK
***-***-7890 ***-***-7890 OK


**Code nộp LeetCode (Cách 4):**

In [16]:
class SolutionTable:
    PREFIX = ["", "+*-", "+**-", "+***-"]
    REMOVE = str.maketrans("", "", "+-() ")

    def maskPII(self, s: str) -> str:
        if "@" in s:
            name, at, domain = s.lower().partition("@")
            return name[0] + "*****" + name[-1] + at + domain
        digits = s.translate(self.REMOVE)
        return self.PREFIX[len(digits) - 10] + "***-***-" + digits[-4:]

print(SolutionTable().maskPII("LeetCode@LeetCode.com"))   # l*****e@leetcode.com

l*****e@leetcode.com


**Độ phức tạp (Cách 4):** Time `O(n)`, Space `O(n)`. Nếu số chữ số vượt 13 (vi phạm ràng buộc), `PREFIX[...]` sẽ báo `IndexError` — một ưu điểm: bảng tra **phát hiện đầu vào sai** thay vì im lặng cho kết quả sai.

## 8. TRACE từng bước

Trace Cách 2 trên ba ví dụ. Ta in từng bước: nhánh nào, giá trị trung gian, kết quả.

| Ví dụ | `"@" in s`? | tên / miền | chữ số | `n = len - 10` | kết quả |
|---|---|---|---|---|---|
| `LeetCode@LeetCode.com` | có | `leetcode` / `leetcode.com` | — | — | `l*****e@leetcode.com` |
| `AB@qq.com` | có | `ab` / `qq.com` | — | — | `a*****b@qq.com` |
| `1(234)567-890` | không | — | `1234567890` | `0` | `***-***-7890` |

Hàm `trace_mask` in từng bước cho một chuỗi bất kỳ.

In [17]:
def trace_mask(s):
    print(f"s = {s!r}")
    if "@" in s:
        low = s.lower()
        name, domain = low.split("@")
        print(f"  nhánh EMAIL: lower -> {low!r}")
        print(f"  name = {name!r} (dài {len(name)}), domain = {domain!r}")
        print(f"  name[0] = {name[0]!r}, name[-1] = {name[-1]!r}; phần giữa {name[1:-1]!r} bị thay bằng '*****'")
        result = name[0] + "*****" + name[-1] + "@" + domain
    else:
        digits = "".join(c for c in s if c.isdigit())
        n = len(digits) - 10
        print(f"  nhánh SỐ ĐIỆN THOẠI: chữ số = {digits!r} (dài {len(digits)})")
        print(f"  n = {len(digits)} - 10 = {n} (độ dài mã quốc gia), mã quốc gia = {digits[:n]!r}")
        print(f"  4 chữ số cuối = {digits[-4:]!r}")
        prefix = "" if n == 0 else "+" + "*" * n + "-"
        print(f"  tiền tố = {prefix!r}")
        result = prefix + "***-***-" + digits[-4:]
    print("  kết quả =", repr(result))
    return result

for s, _ in examples:
    trace_mask(s)

s = 'LeetCode@LeetCode.com'
  nhánh EMAIL: lower -> 'leetcode@leetcode.com'
  name = 'leetcode' (dài 8), domain = 'leetcode.com'
  name[0] = 'l', name[-1] = 'e'; phần giữa 'eetcod' bị thay bằng '*****'
  kết quả = 'l*****e@leetcode.com'
s = 'AB@qq.com'
  nhánh EMAIL: lower -> 'ab@qq.com'
  name = 'ab' (dài 2), domain = 'qq.com'
  name[0] = 'a', name[-1] = 'b'; phần giữa '' bị thay bằng '*****'
  kết quả = 'a*****b@qq.com'
s = '1(234)567-890'
  nhánh SỐ ĐIỆN THOẠI: chữ số = '1234567890' (dài 10)
  n = 10 - 10 = 0 (độ dài mã quốc gia), mã quốc gia = ''
  4 chữ số cuối = '7890'
  tiền tố = ''
  kết quả = '***-***-7890'


Trace thứ hai: các số điện thoại có mã quốc gia 1, 2, 3 chữ số và nhiều kiểu ký tự phân cách. Chú ý chữ số `0` đầu tiên và các khoảng trắng bị bỏ.

In [18]:
for s in ["+1 (234) 567-8901", "86-(10)6769 2020", "+(501) 509-345 66 22"]:
    trace_mask(s)

s = '+1 (234) 567-8901'
  nhánh SỐ ĐIỆN THOẠI: chữ số = '12345678901' (dài 11)
  n = 11 - 10 = 1 (độ dài mã quốc gia), mã quốc gia = '1'
  4 chữ số cuối = '8901'
  tiền tố = '+*-'
  kết quả = '+*-***-***-8901'
s = '86-(10)6769 2020'
  nhánh SỐ ĐIỆN THOẠI: chữ số = '861067692020' (dài 12)
  n = 12 - 10 = 2 (độ dài mã quốc gia), mã quốc gia = '86'
  4 chữ số cuối = '2020'
  tiền tố = '+**-'
  kết quả = '+**-***-***-2020'
s = '+(501) 509-345 66 22'
  nhánh SỐ ĐIỆN THOẠI: chữ số = '5015093456622' (dài 13)
  n = 13 - 10 = 3 (độ dài mã quốc gia), mã quốc gia = '501'
  4 chữ số cuối = '6622'
  tiền tố = '+***-'
  kết quả = '+***-***-***-6622'


## 9. Code nộp LeetCode

Lời giải chuẩn: Cách 2 (hàm chuỗi dựng sẵn) — ngắn gọn, dễ đọc. Đúng `class Solution` và signature.

In [19]:
class Solution:
    def maskPII(self, s: str) -> str:
        if "@" in s:                                          # email
            name, domain = s.lower().split("@")
            return name[0] + "*****" + name[-1] + "@" + domain
        digits = "".join(c for c in s if c.isdigit())         # số điện thoại: chỉ giữ chữ số
        n = len(digits) - 10                                  # độ dài mã quốc gia: 0..3
        prefix = "" if n == 0 else "+" + "*" * n + "-"
        return prefix + "***-***-" + digits[-4:]

sol = Solution()
for s, expected in examples:
    got = sol.maskPII(s)
    print(repr(got), repr(expected), "OK" if got == expected else "SAI")

'l*****e@leetcode.com' 'l*****e@leetcode.com' OK
'a*****b@qq.com' 'a*****b@qq.com' OK
'***-***-7890' '***-***-7890' OK


## 10. Vì sao đúng

Bài này là **mô phỏng quy tắc**, nên "vì sao đúng" là đối chiếu từng quy tắc của đề với từng dòng code.

**Phân loại.** Đề đảm bảo `s` hợp lệ. Email luôn chứa đúng một `@`; số điện thoại chỉ chứa chữ số và `+ - ( )` và khoảng trắng, nên không chứa `@`. Do đó `"@" in s` phân loại **chính xác**.

**Email.**

- Đề: đổi chữ hoa thành chữ thường ở tên và miền → `s.lower()` (đổi toàn chuỗi; `@` và `.` không bị ảnh hưởng).
- Đề: giữ chữ đầu và chữ cuối của tên, thay giữa bằng 5 dấu `*` → `name[0] + "*****" + name[-1]`. Tên có **ít nhất 2 chữ** nên `name[0]` và `name[-1]` là **hai ký tự khác vị trí** (không trùng nhau; chỉ số 0 và `len-1 ≥ 1`).
- Miền giữ nguyên (đã chữ thường).

**Số điện thoại.**

- Đề: bỏ ký tự phân cách → giữ lại đúng các chữ số (vì mọi ký tự khác đều là phân cách).
- Số chữ số `L ∈ [10, 13]`; 10 chữ số cuối là số nội hạt; mã quốc gia có `n = L − 10 ∈ [0, 3]` chữ số.
- Dạng kết quả: `n = 0` → `***-***-XXXX`; `n ≥ 1` → `+` + `n` dấu `*` + `-` + `***-***-XXXX`. Đúng bốn dạng của đề. `XXXX` = 4 chữ số cuối của số nội hạt = 4 chữ số cuối của toàn bộ `digits`, tức `digits[-4:]`.

Để chắc chắn, ta kiểm chứng bằng **oracle dựng từ các mảnh đã biết**: sinh ngẫu nhiên email từ `(tên, miền)` hoặc số điện thoại từ `(chuỗi chữ số, ký tự phân cách chèn ngẫu nhiên)`, tính đáp án mong đợi **trực tiếp từ các mảnh** (không qua việc phân tích chuỗi), rồi so với `Solution`.

In [20]:
LETTERS = "abcdefghijklmnopqrstuvwxyzABCDEFGHIJKLMNOPQRSTUVWXYZ"

def random_email(rng):
    # sinh (chuỗi, đáp án mong đợi) hợp lệ theo ràng buộc: 8 <= len <= 40
    while True:
        name_len = rng.randint(2, 20)
        dom_a, dom_b = rng.randint(1, 8), rng.randint(1, 8)
        name = "".join(rng.choice(LETTERS) for _ in range(name_len))
        dom1 = "".join(rng.choice(LETTERS) for _ in range(dom_a))
        dom2 = "".join(rng.choice(LETTERS) for _ in range(dom_b))
        s = f"{name}@{dom1}.{dom2}"
        if 8 <= len(s) <= 40:
            expected = name[0].lower() + "*****" + name[-1].lower() + "@" + (dom1 + "." + dom2).lower()
            return s, expected

def random_phone(rng):
    seps = "+-() "
    while True:
        total = rng.randint(10, 13)
        digits = "".join(rng.choice("0123456789") for _ in range(total))
        # chèn ký tự phân cách ngẫu nhiên giữa/đầu/cuối các chữ số (không dính nhau thì thôi cũng được)
        out = []
        for ch in digits:
            if rng.random() < 0.35:
                out.append(rng.choice(seps))
            out.append(ch)
        if rng.random() < 0.3:
            out.append(rng.choice(seps))
        s = "".join(out)
        if 10 <= len(s) <= 20:
            n = total - 10
            expected = ("" if n == 0 else "+" + "*" * n + "-") + "***-***-" + digits[-4:]
            return s, expected

rng = random.Random(831)
for _ in range(2000):
    s, expected = random_email(rng)
    assert Solution().maskPII(s) == expected, (s, expected)
    s, expected = random_phone(rng)
    assert Solution().maskPII(s) == expected, (s, expected)
print("4000 input sinh từ các mảnh đã biết: Solution khớp đáp án dựng trực tiếp")

4000 input sinh từ các mảnh đã biết: Solution khớp đáp án dựng trực tiếp


## 11. Time / Space Complexity

Gọi `n = len(s)` (tối đa 40).

- **Time:** `O(n)` — `lower`, `split`, lọc chữ số, cắt chuỗi đều tuyến tính theo độ dài.
- **Space:** `O(n)` — chuỗi trung gian (`digits`, `name`, `domain`, kết quả). Với `n ≤ 40` đây là hằng số rất nhỏ.

Bài không có yếu tố làm chậm: mọi cách đều chạy tức thì. Điểm khác biệt thật sự giữa các cách là **độ ngắn gọn, độ dễ đọc và khả năng bảo trì**.

In [21]:
import time
impls = {
    "manual": mask_pii_manual,
    "builtin": mask_pii_builtin,
    "regex": mask_pii_regex,
    "table": mask_pii_table,
}
rng = random.Random(1)
cases = [random_email(rng)[0] for _ in range(300)] + [random_phone(rng)[0] for _ in range(300)]
for name, f in impls.items():
    t0 = time.perf_counter()
    for _ in range(20):
        for s in cases:
            f(s)
    dt = time.perf_counter() - t0
    print(f"{name:<8} {20 * len(cases)} lời gọi xong")
print("(thời gian không in ra vì phụ thuộc máy; tất cả đều rất nhanh)")

manual   12000 lời gọi xong


builtin  12000 lời gọi xong


regex    12000 lời gọi xong


table    12000 lời gọi xong
(thời gian không in ra vì phụ thuộc máy; tất cả đều rất nhanh)


## 12. Bộ test

Gồm: ví dụ đề, edge case, phản ví dụ cho các lỗi thường gặp, đối chiếu các cách bằng `assert`, và test ngẫu nhiên có seed đúng ràng buộc.

In [22]:
sol = Solution()

# 1) Ví dụ đề
assert sol.maskPII("LeetCode@LeetCode.com") == "l*****e@leetcode.com"
assert sol.maskPII("AB@qq.com") == "a*****b@qq.com"
assert sol.maskPII("1(234)567-890") == "***-***-7890"
print("ví dụ đề OK")

# 2) Edge case email
assert sol.maskPII("aB@a.b") == "a*****b@a.b"                       # (độ dài 6 < 8 là ngoài ràng buộc nhưng quy tắc vẫn đúng)
assert sol.maskPII("Ab@Cd.eF") == "a*****b@cd.ef"                   # hoa/thường lẫn lộn ở miền
assert sol.maskPII("ZZZZZZZZZZ@ZZ.ZZ") == "z*****z@zz.zz"           # lặp ký tự: phần giữa vẫn chỉ 5 sao
assert sol.maskPII("aaa@aaa.aaa") == "a*****a@aaa.aaa"
assert sol.maskPII("ab@cd.ef") == "a*****b@cd.ef"
assert sol.maskPII("Xy@qq.COM") == "x*****y@qq.com"
print("edge case email OK")

# 3) Edge case phone: mã quốc gia 0, 1, 2, 3 chữ số, và nhiều ký tự phân cách
assert sol.maskPII("1234567890") == "***-***-7890"                   # không ký tự phân cách
assert sol.maskPII("(123) 456-7890") == "***-***-7890"               # bắt đầu bằng '('
assert sol.maskPII("+1 (234) 567-8901") == "+*-***-***-8901"         # bắt đầu bằng '+'; 1 chữ số mã
assert sol.maskPII("86-(10)6769 2020") == "+**-***-***-2020"         # 2 chữ số mã
assert sol.maskPII("+(501) 509-345 66 22") == "+***-***-***-6622"     # 13 chữ số: mã quốc gia 3 chữ số
assert sol.maskPII("+86(10)69 55 01 23") == "+**-***-***-0123"
assert sol.maskPII("0000000000") == "***-***-0000"                   # toàn số 0
assert sol.maskPII("+++1-(234)-567-8901") == "+*-***-***-8901"       # nhiều ký tự phân cách liên tiếp
print("edge case phone OK")

ví dụ đề OK
edge case email OK
edge case phone OK


In [23]:
# 4) Phản ví dụ cho các lỗi thường gặp (xem mục 13)
def wrong_stars_by_length(s):
    name, domain = s.lower().split("@")
    return name[0] + "*" * (len(name) - 2) + name[-1] + "@" + domain       # SAI: số sao theo độ dài

assert wrong_stars_by_length("AB@qq.com") != sol.maskPII("AB@qq.com")
assert wrong_stars_by_length("LeetCode@LeetCode.com") != sol.maskPII("LeetCode@LeetCode.com")
print("lỗi 'số sao theo độ dài tên' bị bắt bởi ví dụ 1 và 2")

# 5) Đối chiếu 4 cách tự viết + 4 class trên input ngẫu nhiên có seed (đúng ràng buộc)
classes = [SolutionBuiltin(), SolutionRegex(), SolutionTable(), Solution()]
funcs = [mask_pii_manual, mask_pii_builtin, mask_pii_regex, mask_pii_table]
rng = random.Random(2026)
for _ in range(1500):
    for gen in (random_email, random_phone):
        s, expected = gen(rng)
        for f in funcs:
            assert f(s) == expected, (f.__name__, s)
        for c in classes:
            assert c.maskPII(s) == expected, (type(c).__name__, s)
print("3000 input ngẫu nhiên: mọi cách khớp đáp án dựng từ mảnh")

# 6) Tính chất: độ dài kết quả cố định theo loại
for _ in range(500):
    s, _e = random_email(rng)
    r = sol.maskPII(s)
    name, domain = s.split("@")
    assert len(r) == 1 + 5 + 1 + 1 + len(domain)                  # đầu + 5 sao + cuối + '@' + miền
    assert r == r.lower()
    s, _e = random_phone(rng)
    r = sol.maskPII(s)
    assert len(r) in (12, 15, 16, 17)                               # '***-***-XXXX'=12; thêm 3,4,5 ký tự tiền tố
    assert r.endswith(r[-4:]) and r[-4:].isdigit()
print("test tính chất OK")

lỗi 'số sao theo độ dài tên' bị bắt bởi ví dụ 1 và 2


3000 input ngẫu nhiên: mọi cách khớp đáp án dựng từ mảnh
test tính chất OK


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Số dấu `*` tính theo độ dài tên

Nghĩ rằng "thay các chữ giữa bằng dấu sao" tức là mỗi chữ một sao. Nhưng đề yêu cầu **đúng 5 dấu sao** bất kể độ dài.

In [24]:
def wrong_stars_per_letter(s):
    name, domain = s.lower().split("@")
    return name[0] + "*" * (len(name) - 2) + name[-1] + "@" + domain      # SAI

print("sai :", wrong_stars_per_letter("AB@qq.com"), "(kỳ vọng a*****b@qq.com)")
print("sai :", wrong_stars_per_letter("LeetCode@LeetCode.com"), "(kỳ vọng l*****e@leetcode.com)")
print("đúng:", Solution().maskPII("AB@qq.com"), Solution().maskPII("LeetCode@LeetCode.com"))

sai : ab@qq.com (kỳ vọng a*****b@qq.com)
sai : l******e@leetcode.com (kỳ vọng l*****e@leetcode.com)
đúng: a*****b@qq.com l*****e@leetcode.com


**Vì sao sai:** với tên 2 chữ ta được 0 sao; với tên 8 chữ ta được 6 sao. Đề luôn là 5. **Quy tắc:** đọc kỹ con số cụ thể trong đề.

### Lỗi 2 — Quên đổi chữ thường (chỉ đổi tên hoặc chỉ đổi chữ đầu)

In [25]:
def wrong_no_lower(s):
    name, domain = s.split("@")
    return name[0] + "*****" + name[-1] + "@" + domain                      # SAI: thiếu lower

print("sai :", wrong_no_lower("LeetCode@LeetCode.com"), "(kỳ vọng l*****e@leetcode.com)")
print("đúng:", Solution().maskPII("LeetCode@LeetCode.com"))

sai : L*****e@LeetCode.com (kỳ vọng l*****e@leetcode.com)
đúng: l*****e@leetcode.com


**Vì sao sai:** đề yêu cầu cả tên **và miền** thành chữ thường; chữ đầu/cuối của tên cũng phải thường. **Cách đúng:** `s.lower()` ngay từ đầu nhánh email.

### Lỗi 3 — Lấy 4 chữ số cuối trước khi bỏ ký tự phân cách

`s[-4:]` lấy 4 **ký tự** cuối, không phải 4 **chữ số** cuối.

In [26]:
def wrong_last4_raw(s):
    digits = "".join(c for c in s if c.isdigit())
    n = len(digits) - 10
    prefix = "" if n == 0 else "+" + "*" * n + "-"
    return prefix + "***-***-" + s[-4:]                                     # SAI: s[-4:] thay vì digits[-4:]

print("sai :", wrong_last4_raw("1(234)567-890"), "(kỳ vọng ***-***-7890)")
print("đúng:", Solution().maskPII("1(234)567-890"))
print("sai :", wrong_last4_raw("(123) 456-7890 "), "| đúng:", Solution().maskPII("(123) 456-7890 "))

sai : ***-***--890 (kỳ vọng ***-***-7890)
đúng: ***-***-7890
sai : ***-***-890  | đúng: ***-***-7890


**Vì sao sai:** `"1(234)567-890"[-4:]` là `"-890"`. **Quy tắc:** bỏ ký tự phân cách **trước**, rồi mới cắt.

### Lỗi 4 — Tính sai độ dài mã quốc gia hoặc quên `+`, `-`

Các lỗi hay gặp: dùng `len(digits) - 11`, quên chữ `+`, quên dấu `-` sau các sao, hay dùng bảng dạng `"***"` thay vì `"+***-"`.

In [27]:
def wrong_country_off_by_one(s):
    digits = "".join(c for c in s if c.isdigit())
    n = len(digits) - 11                                                    # SAI: phải trừ 10
    prefix = "" if n <= 0 else "+" + "*" * n + "-"
    return prefix + "***-***-" + digits[-4:]

def wrong_no_plus(s):
    digits = "".join(c for c in s if c.isdigit())
    n = len(digits) - 10
    prefix = "" if n == 0 else "*" * n + "-"                                # SAI: thiếu '+'
    return prefix + "***-***-" + digits[-4:]

s = "+1 (234) 567-8901"
print("sai :", wrong_country_off_by_one(s))
print("sai :", wrong_no_plus(s))
print("đúng:", Solution().maskPII(s))

sai : ***-***-8901
sai : *-***-***-8901
đúng: +*-***-***-8901


**Vì sao sai:** mã quốc gia là phần chữ số **trước 10 chữ số cuối**: `len(digits) - 10`. Và dạng kết quả có dấu `+` ở đầu, dấu `-` ngăn mọi nhóm. **Cách tránh:** viết bảng `PREFIX = ["", "+*-", "+**-", "+***-"]` rồi đối chiếu từng dòng với đề.

### Lỗi 5 — Nhận diện số điện thoại bằng `s[0].isdigit()`

Số điện thoại có thể bắt đầu bằng `+` hay `(`, nên điều kiện này sai.

In [28]:
def wrong_detect(s):
    if s[0].isdigit():                                  # SAI: chỉ nhận ra số điện thoại bắt đầu bằng chữ số
        digits = "".join(c for c in s if c.isdigit())
        n = len(digits) - 10
        prefix = "" if n == 0 else "+" + "*" * n + "-"
        return prefix + "***-***-" + digits[-4:]
    name, domain = s.lower().split("@")                 # số điện thoại '(123) ...' rơi vào đây -> lỗi
    return name[0] + "*****" + name[-1] + "@" + domain

for s in ["1234567890", "(123) 456-7890"]:
    try:
        print(repr(s), "->", wrong_detect(s))
    except ValueError as e:
        print(repr(s), "-> LỖI:", type(e).__name__)
print("đúng:", Solution().maskPII("(123) 456-7890"))

'1234567890' -> ***-***-7890
'(123) 456-7890' -> LỖI: ValueError
đúng: ***-***-7890


**Vì sao sai:** `"(123) 456-7890"` bắt đầu bằng `(`, không phải chữ số → bị đưa nhầm vào nhánh email. Dùng `"@" in s` (email luôn có `@`, số điện thoại không bao giờ có).

### Lỗi 6 — Dùng `str.replace` để che phần giữa tên

`name.replace(name[1:-1], "*****")` thay **mọi** chỗ xuất hiện của chuỗi giữa, không chỉ chỗ ở giữa.

In [29]:
def wrong_replace(s):
    name, domain = s.lower().split("@")
    masked = name.replace(name[1:-1], "*****")           # SAI
    return masked + "@" + domain

print("sai :", wrong_replace("AAA@qq.com"), "(kỳ vọng a*****a@qq.com)")
print("sai :", wrong_replace("abcabc@qq.com"), "(kỳ vọng a*****c@qq.com)")
print("đúng:", Solution().maskPII("AAA@qq.com"), Solution().maskPII("abcabc@qq.com"))

sai : ***************@qq.com (kỳ vọng a*****a@qq.com)
sai : a*****c@qq.com (kỳ vọng a*****c@qq.com)
đúng: a*****a@qq.com a*****c@qq.com


**Vì sao sai:** với tên `aaa`, phần giữa là `a`; `replace("a", "*****")` thay **cả ba** chữ `a`. **Cách đúng:** không cần thay thế — cứ ghép `name[0] + "*****" + name[-1]`.

## 14. Mẹo & nhận dạng pattern

- **Bài "mô phỏng quy tắc" xử lý chuỗi** → đọc kỹ đề, viết thành các bước nhỏ, **kiểm tra từng ví dụ bằng tay** trước khi code. Con số hằng (ở đây "5 sao", "10 chữ số", "4 chữ số cuối") nên viết thành hằng số hoặc bảng.
- **Rẽ nhánh bằng đặc trưng bền vững nhất:** `'@' in s` hơn là suy luận từ ký tự đầu.
- **Lọc ký tự không mong muốn trước, tính sau:** bỏ ký tự phân cách rồi mới đếm/cắt (cùng kiểu với bài Valid Palindrome, Reformat Phone Number 1694...).
- **Bảng tra thay `if/elif`:** khi kết quả chỉ phụ thuộc một số nhỏ (độ dài mã quốc gia 0..3).
- **Chuỗi bất biến:** mỗi phép biến đổi trả chuỗi **mới**; muốn che thì **dựng** chuỗi mới từ các mảnh.
- **Regex vừa đủ:** dùng để *tách mảnh* hoặc *xoá ký tự*; đừng nhồi cả logic vào một mẫu khó đọc.
- Bài họ hàng: Reformat Phone Number (1694), Defanging an IP Address (1108), Valid Number (65), Goal Parser Interpretation (1678).

## 15. Biến thể

### 15.1. Kiểm tra hợp lệ rồi mới che

Trong thực tế đầu vào có thể sai. Hàm dưới trả về `None` nếu chuỗi không phải email/số điện thoại hợp lệ theo đề, ngược lại trả về kết quả che. Dùng `re.fullmatch` với mẫu mô tả đúng quy tắc: tên ≥ 2 chữ cái, miền = chữ + `.` + chữ; số điện thoại: chỉ ký tự trong tập cho phép và 10–13 chữ số.

In [30]:
EMAIL_RE = re.compile(r"[A-Za-z]{2,}@[A-Za-z]+\.[A-Za-z]+")
PHONE_CHARS_RE = re.compile(r"[0-9+\-() ]+")

def mask_pii_checked(s):
    if EMAIL_RE.fullmatch(s):
        return Solution().maskPII(s)
    if PHONE_CHARS_RE.fullmatch(s):
        count = sum(c.isdigit() for c in s)
        if 10 <= count <= 13:
            return Solution().maskPII(s)
    return None

for s in ["AB@qq.com", "A@qq.com", "AB@qqcom", "AB@.com", "123", "(123) 456-7890", "12-34", "abc", "+1 (234) 567-8901"]:
    print(f"{s!r:>22} -> {mask_pii_checked(s)!r}")

           'AB@qq.com' -> 'a*****b@qq.com'
            'A@qq.com' -> None
            'AB@qqcom' -> None
             'AB@.com' -> None
                 '123' -> None
      '(123) 456-7890' -> '***-***-7890'
               '12-34' -> None
                 'abc' -> None
   '+1 (234) 567-8901' -> '+*-***-***-8901'


### 15.2. Che mọi email/số điện thoại trong một đoạn văn bản

Dùng `re.sub` với hàm thay thế: tìm mọi chuỗi giống email (`[A-Za-z]{2,}@[A-Za-z]+\.[A-Za-z]+`) và thay bằng bản đã che. (Số điện thoại trong văn bản phức tạp hơn vì chứa khoảng trắng; ở đây chỉ làm email để thấy kỹ thuật.)

In [31]:
EMAIL_IN_TEXT = re.compile(r"[A-Za-z]{2,}@[A-Za-z]+\.[A-Za-z]+")

def mask_emails_in_text(text):
    return EMAIL_IN_TEXT.sub(lambda m: Solution().maskPII(m.group(0)), text)

text = "Liên hệ Alice@Example.com hoặc bob@mail.org; không phải x@y.z hay a@b."
print(mask_emails_in_text(text))

Liên hệ a*****e@example.com hoặc b*****b@mail.org; không phải x@y.z hay a@b.


### 15.3. Số dấu sao và số chữ số giữ lại tuỳ chỉnh

Tham số hoá: `stars` (số dấu `*` trong email), `keep` (số chữ số cuối giữ lại). Số điện thoại che dạng `"*" * (len(digits) - keep)` + `digits[-keep:]`, nhóm lại theo quy tắc gốc khi `keep = 4`.

In [32]:
def mask_pii_custom(s, stars=5, keep=4):
    if "@" in s:
        name, domain = s.lower().split("@")
        return name[0] + "*" * stars + name[-1] + "@" + domain
    digits = "".join(c for c in s if c.isdigit())
    return "*" * (len(digits) - keep) + digits[len(digits) - keep:]

print(mask_pii_custom("LeetCode@LeetCode.com", stars=3))      # l***e@leetcode.com
print(mask_pii_custom("1(234)567-890", keep=2))               # ********90
print(mask_pii_custom("AB@qq.com") == Solution().maskPII("AB@qq.com"))

l***e@leetcode.com
********90
True


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Duyệt từng ký tự | Tự dùng vòng lặp, `ord/chr`, `if/elif` | `O(n)` | `O(n)` | Chỉ để hiểu từng hàm dựng sẵn thay thế đoạn nào |
| 2. Hàm chuỗi dựng sẵn | `lower`, `split`, `isdigit`, nhân chuỗi | `O(n)` | `O(n)` | **Lời giải chuẩn nộp LeetCode**: ngắn, dễ đọc |
| 3. Biểu thức chính quy | `fullmatch` tách nhóm; `sub(\D)` xoá | `O(n)` | `O(n)` | Khi cần *mô tả* dạng hợp lệ / mở rộng xử lý văn bản |
| 4. Bảng tra + `translate` | `PREFIX[n]`, xoá phân cách một lệnh | `O(n)` | `O(n)` | Quy tắc nằm trong bảng; tự báo lỗi nếu vượt 13 chữ số |

## 17. Bài tập vận dụng

Làm tuần tự từ dễ đến khó. Các cell có chữ `TODO` **không được gọi** khi chạy notebook; bạn tự điền rồi bỏ dấu `#` ở các dòng `assert` để kiểm tra. Đáp án nằm ở mục 19.

### Bài 1 — Đoán output (dễ)

Không chạy code, hãy đoán output của ba dòng sau, rồi chạy kiểm tra:

```python
print(Solution().maskPII("Zz@Ab.cD"))
print(Solution().maskPII("(555) 123-4567"))
print(Solution().maskPII("+7 495 123 45 67"))
```

In [33]:
# Bài 1: viết dự đoán của bạn vào các biến bên dưới (chưa chạy code mẫu)
my_guess_line1 = None
my_guess_line2 = None
my_guess_line3 = None

### Bài 2 — Tính tay (dễ-vừa)

Với `s = "86-(10)6769 2020"`: hãy cho biết (a) chuỗi chữ số, (b) độ dài mã quốc gia, (c) kết quả che. Điền vào các biến.

In [34]:
# Bài 2: điền kết quả tính tay
hand_digits = None
hand_country_len = None
hand_result = None

### Bài 3 — Điền chỗ trống (vừa)

Hoàn thành nhánh **số điện thoại** của lời giải.

In [35]:
def exercise_3_mask_phone(s: str) -> str:
    # TODO 1: digits = chuỗi chỉ gồm các chữ số của s
    # TODO 2: n = độ dài mã quốc gia (len(digits) - 10)
    # TODO 3: prefix = "" nếu n == 0, ngược lại "+" + n dấu '*' + "-"
    # TODO 4: return prefix + "***-***-" + 4 chữ số cuối
    return None


# assert exercise_3_mask_phone("1(234)567-890") == "***-***-7890"
# assert exercise_3_mask_phone("+1 (234) 567-8901") == "+*-***-***-8901"
# assert exercise_3_mask_phone("86-(10)6769 2020") == "+**-***-***-2020"

### Bài 4 — Tự viết lại (vừa-khó)

Không nhìn lại Cách 3, hãy tự viết `exercise_4_regex(s)` giải **cả hai nhánh** bằng `re`: nhánh email dùng `re.fullmatch` với 3 nhóm; nhánh số điện thoại dùng `re.sub(r"\D", "", s)`.

In [36]:
def exercise_4_regex(s: str) -> str:
    # TODO
    pass


# assert exercise_4_regex("LeetCode@LeetCode.com") == "l*****e@leetcode.com"
# assert exercise_4_regex("AB@qq.com") == "a*****b@qq.com"
# assert exercise_4_regex("+86 (10) 6955-0123") == "+**-***-***-0123"

### Bài 5 — Mini challenge: nhận dạng loại chuỗi (khó)

Viết `exercise_5_kind(s)` trả về `"email"`, `"phone"` hoặc `None` nếu `s` **không hợp lệ** theo đúng các quy tắc của đề (không phải ràng buộc độ dài): email = tên ≥ 2 chữ cái + `@` + (chữ cái)+ `.` (chữ cái)+; số điện thoại = chỉ gồm chữ số và các ký tự `+ - ( )` hoặc khoảng trắng, với **10 đến 13 chữ số**.

```python
exercise_5_kind("AB@qq.com")        -> "email"
exercise_5_kind("A@qq.com")         -> None      # tên chỉ 1 chữ
exercise_5_kind("(123) 456-7890")   -> "phone"
exercise_5_kind("12345")            -> None      # chỉ 5 chữ số
exercise_5_kind("1" * 14)           -> None      # 14 chữ số
```

In [37]:
def exercise_5_kind(s: str):
    # TODO
    pass


# assert exercise_5_kind("AB@qq.com") == "email"
# assert exercise_5_kind("A@qq.com") is None
# assert exercise_5_kind("(123) 456-7890") == "phone"
# assert exercise_5_kind("12345") is None

## 18. Gợi ý 3 mức

**Bài 3 / Bài 4**

- *Mức 1 (nhẹ):* Phân loại bằng `"@" in s`. Số điện thoại: bỏ hết ký tự không phải chữ số trước tiên.
- *Mức 2 (vừa):* Sau khi có `digits`: mã quốc gia dài `len(digits) - 10`; 4 chữ số cuối là `digits[-4:]`. Tiền tố là rỗng khi `n == 0`.
- *Mức 3 (gần đáp án):* `prefix = "" if n == 0 else "+" + "*" * n + "-"`; trả `prefix + "***-***-" + digits[-4:]`. Regex email: `([A-Za-z])[A-Za-z]*([A-Za-z])@(.+)`, ghép `first + "*****" + last + "@" + domain` rồi `.lower()`.

**Bài 5**

- *Mức 1:* Hai loại có đặc điểm khác hẳn nhau; thử khớp từng loại.
- *Mức 2:* Email: `re.fullmatch(r"[A-Za-z]{2,}@[A-Za-z]+\.[A-Za-z]+", s)`. Số điện thoại: kiểm tra mọi ký tự thuộc tập cho phép, đếm chữ số.
- *Mức 3:* Phone: `all(c in "0123456789+-() " for c in s)` và `10 <= sum(c.isdigit() for c in s) <= 13`. Đừng quên rằng chuỗi rỗng/chỉ ký hiệu có 0 chữ số → `None`.

## 19. Đáp án tham khảo

### Đáp án Bài 1

- `Zz@Ab.cD` → `z*****z@ab.cd`.
- `(555) 123-4567` → `***-***-4567`.
- `+7 495 123 45 67` → chữ số `74951234567` (11 chữ số, mã quốc gia 1) → `+*-***-***-4567`.

In [38]:
assert Solution().maskPII("Zz@Ab.cD") == "z*****z@ab.cd"
assert Solution().maskPII("(555) 123-4567") == "***-***-4567"
assert Solution().maskPII("+7 495 123 45 67") == "+*-***-***-4567"
print("Bài 1 OK")

Bài 1 OK


### Đáp án Bài 2

`"86-(10)6769 2020"`: chữ số Ghép các chữ số theo thứ tự: `86`, `10`, `6769`, `2020` → `861067692020` (12 chữ số). Mã quốc gia `12 − 10 = 2` chữ số (`86`). Kết quả `+**-***-***-2020`.

In [39]:
hand_digits = "861067692020"
hand_country_len = 2
hand_result = "+**-***-***-2020"
assert hand_digits == "".join(c for c in "86-(10)6769 2020" if c.isdigit())
assert hand_country_len == len(hand_digits) - 10
assert hand_result == Solution().maskPII("86-(10)6769 2020")
print("Bài 2 OK")

Bài 2 OK


### Đáp án Bài 3

In [40]:
def exercise_3_solution(s: str) -> str:
    digits = "".join(c for c in s if c.isdigit())
    n = len(digits) - 10
    prefix = "" if n == 0 else "+" + "*" * n + "-"
    return prefix + "***-***-" + digits[-4:]

assert exercise_3_solution("1(234)567-890") == "***-***-7890"
assert exercise_3_solution("+1 (234) 567-8901") == "+*-***-***-8901"
assert exercise_3_solution("86-(10)6769 2020") == "+**-***-***-2020"
rng = random.Random(3)
for _ in range(300):
    s, expected = random_phone(rng)
    assert exercise_3_solution(s) == expected
print("Bài 3 OK")

Bài 3 OK


### Đáp án Bài 4

In [41]:
def exercise_4_solution(s: str) -> str:
    m = re.fullmatch(r"([A-Za-z])[A-Za-z]*([A-Za-z])@(.+)", s)
    if m:
        first, last, domain = m.groups()
        return f"{first}*****{last}@{domain}".lower()
    digits = re.sub(r"\D", "", s)
    n = len(digits) - 10
    prefix = "" if n == 0 else "+" + "*" * n + "-"
    return prefix + "***-***-" + digits[-4:]

assert exercise_4_solution("LeetCode@LeetCode.com") == "l*****e@leetcode.com"
assert exercise_4_solution("AB@qq.com") == "a*****b@qq.com"
assert exercise_4_solution("+86 (10) 6955-0123") == "+**-***-***-0123"
rng = random.Random(4)
for _ in range(300):
    for gen in (random_email, random_phone):
        s, expected = gen(rng)
        assert exercise_4_solution(s) == expected
print("Bài 4 OK")

Bài 4 OK


### Đáp án Bài 5

In [42]:
def exercise_5_solution(s: str):
    if re.fullmatch(r"[A-Za-z]{2,}@[A-Za-z]+\.[A-Za-z]+", s):
        return "email"
    if s and all(c in "0123456789+-() " for c in s) and 10 <= sum(c.isdigit() for c in s) <= 13:
        return "phone"
    return None

assert exercise_5_solution("AB@qq.com") == "email"
assert exercise_5_solution("A@qq.com") is None
assert exercise_5_solution("(123) 456-7890") == "phone"
assert exercise_5_solution("12345") is None
assert exercise_5_solution("1" * 14) is None
assert exercise_5_solution("1" * 13) == "phone"
assert exercise_5_solution("1" * 10) == "phone"
assert exercise_5_solution("") is None
assert exercise_5_solution("a1b") is None

# đối chiếu với mask_pii_checked (cách làm khác) trên chuỗi ngẫu nhiên
rng = random.Random(55)
pool = "ab@.Z0123456789+-() x"
for _ in range(3000):
    s = "".join(rng.choice(pool) for _ in range(rng.randint(0, 22)))
    kind = exercise_5_solution(s)
    assert (kind is not None) == (mask_pii_checked(s) is not None), repr(s)
print("Bài 5 OK")

Bài 5 OK


## 20. Template nhớ nhanh + Checklist trước khi submit

### Template

```python
class Solution:
    def maskPII(self, s: str) -> str:
        if "@" in s:                                           # email
            name, domain = s.lower().split("@")
            return name[0] + "*****" + name[-1] + "@" + domain
        digits = "".join(c for c in s if c.isdigit())          # số điện thoại
        n = len(digits) - 10                                   # độ dài mã quốc gia
        prefix = "" if n == 0 else "+" + "*" * n + "-"
        return prefix + "***-***-" + digits[-4:]
```

Câu thần chú: **"Có `@` là email: lower, giữ đầu cuối tên, đúng 5 sao. Không có là điện thoại: giữ chữ số, mã quốc gia = dài − 10, 4 số cuối."**

### Checklist trước khi submit

- [ ] Phân loại bằng `"@" in s` (không dùng `s[0].isdigit()`).
- [ ] Email: `lower()` cho cả tên và miền; **đúng 5 dấu `*`**, kể cả khi tên chỉ 2 chữ.
- [ ] Điện thoại: bỏ ký tự phân cách **trước** rồi mới cắt `digits[-4:]`.
- [ ] Mã quốc gia `= len(digits) - 10`; tiền tố `+`, đúng `n` dấu `*`, rồi dấu `-`; `n = 0` thì không có tiền tố.
- [ ] Đã thử tên 2 chữ, tên có chữ lặp, số bắt đầu bằng `(` hoặc `+`, mã quốc gia 0/1/2/3 chữ số.
- [ ] Đã đối chiếu với oracle dựng từ các mảnh đã biết.
- [ ] Skeleton đúng: `class Solution`, method `maskPII(self, s: str) -> str`.